In [ ]:
# %% [markdown]
# # SoundSpaces rollout evaluation analysis
#
# Analog of `rollout_loopnav_analysis.ipynb` for a SoundSpaces battery. Loads
# per-(arm, clip) records from `scripts/world/eval_rollout.py` and compares
# four arms: Temporal, WorldMem, FAR -- Meta (`pnp_meta`, metadata scoring
# alone) and FAR -- Multi-Cue (`pnp_amz`, audio + metadata with a z-gated
# fusion). The meta-vs-multi-cue pair isolates the audio channel: the same
# retriever with the audio cue on or off.
#
# Needs: a record battery under `results/rollout_eval/<battery>/<arm>/*.npz`
# (one sub-directory per arm plus `protocol.json`; `EVAL_DIR` below names the
# battery) and the per-clip observability file (`OBS_PATH`, built by
# `scripts/data/build_observability_soundspaces.py`) for the coverage view.
# Runs on the CPU.
#
# Views: (1) SHORT/MIDDLE/LONG per-bin normalized-time curves; (2) full-
# trajectory summary + box plots; (3) end-aligned real-time curves;
# (4) per-clip PAIRED differences vs temporal (Wilcoxon); (5) retrieval
# diagnostics vs progress; (6) SS-SPECIFIC coverage dose-response of the
# audio channel (paired FAR -- Multi-Cue minus FAR -- Meta, against the
# clip's outbound->return view coverage);
# (7) PAPER FIGURE — aggregated normalized-time curves over ALL clips,
# styled to match the LoopNav figure.

# %%
import glob
import json
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

from far.constants import ROOT

os.chdir(ROOT)   # EVAL_DIR is repo-root-relative (Jupyter kernels start in notebooks/)

EVAL_DIR = "results/rollout_eval/ssv2_0700000_stride16_context12"
# Keep OBS_PATH corpus-consistent with EVAL_DIR: v1 batteries -> observability,
# v2 batteries -> observability_v2 (ids are shared, so a mismatch is SILENT).
OBS_PATH = "results/observability_v2/soundspaces_observability.json"
ROUTE    = "both"    # "ABA" | "ABCA" | "both" — restricts ALL downstream views

# Arm labels = the record sub-dirs of EVAL_DIR (the `arms:` keys the battery
# was launched with): v1 batteries use `ss_*`, v2 batteries `ssv2_*`.
ARMS   = ["ssv2_temporal", "ssv2_worldmem", "ssv2_pnp_meta", "ssv2_pnp_amz"]
COLORS = dict(zip(ARMS, ["tab:blue", "tab:orange", "tab:green", "tab:red"]))
# ARMS   = ["ss_temporal", "ss_worldmem", "ss_pnp_meta", "ss_pnp_amz"]
# COLORS = dict(zip(ARMS, ["tab:blue", "tab:orange", "tab:green", "tab:red"]))
# ARMS   = ["ss_pnp_meta", "ss_pnp_amz"]
# COLORS = dict(zip(ARMS, ["tab:green", "tab:red"]))
# ARMS   = ["ssv2_pnp_meta", "ssv2_pnp_amz"]
# COLORS = dict(zip(ARMS, ["tab:green", "tab:red"]))
METRICS = ["psnr", "lpips", "dreamsim"]
BETTER  = {"psnr": "higher is better", "lpips": "lower is better", "dreamsim": "lower is better"}

protocol = json.load(open(f"{EVAL_DIR}/protocol.json"))
STRIDE, FPS = protocol["rollout_stride"], 10.0   # soundspaces fps = 10

_missing = [a for a in ARMS if not os.path.isdir(f"{EVAL_DIR}/{a}")]
assert not _missing, (f"no record dir for arms {_missing} under {EVAL_DIR} — "
                      f"present: {sorted(d for d in os.listdir(EVAL_DIR) if os.path.isdir(f'{EVAL_DIR}/{d}'))}")
recs = {arm: {int(Path(p).stem): dict(np.load(p, allow_pickle=False))
              for p in sorted(glob.glob(f"{EVAL_DIR}/{arm}/*.npz"))}
        for arm in ARMS}
# Paired design: analyse the clips COMPLETED BY EVERY ARM. On a finished
# battery this is all of them; on an in-progress one it is the common subset,
# so the notebook can be re-run while collection continues (results tighten as
# records land — pairing stays exact either way).
clips = sorted(set.intersection(*(set(recs[a]) for a in ARMS)))
assert clips, f"no clips completed by every arm yet under {EVAL_DIR}"
per_arm = {a: len(recs[a]) for a in ARMS}
if any(n != len(clips) for n in per_arm.values()):
    print(f"in-progress battery: per-arm records {per_arm} -> analysing "
          f"{len(clips)} common clips")
for arm in ARMS:
    recs[arm] = {c: recs[arm][c] for c in clips}

# Per-clip observability coverage (outbound->return view support; NaN when
# the clip is unscored). soundspaces ids are flat: <scene>_<route>_<tier>_<k>.
obs = json.load(open(OBS_PATH))
cov_map = {e["video_id"]: e["coverage"] for e in obs["episodes"]}

meta = {}
for c in clips:
    r0 = recs[ARMS[0]][c]
    vid = str(r0["video_id"])
    scene, route, tier = vid.split("_")[:3]
    n = int(r0["n_steps"])
    assert all(int(recs[a][c]["n_steps"]) == n for a in ARMS)
    meta[c] = dict(video_id=vid, scene=scene, route=route, tier=tier,
                   n_steps=n, coverage=cov_map.get(vid, np.nan))
if ROUTE != "both":
    clips = [c for c in clips if meta[c]["route"] == ROUTE]
    assert clips, f"no {ROUTE} clips in the common subset"
    recs = {a: {c: recs[a][c] for c in clips} for a in ARMS}
    meta = {c: meta[c] for c in clips}
    print(f"route filter: analysing {len(clips)} {ROUTE} clips")
cov_all = np.array([meta[c]["coverage"] for c in clips])
print(f"{len(clips)} clips x {len(ARMS)} arms | stride {STRIDE}, "
      f"context_size {protocol['context_size']}, ddim{protocol['sampling_steps']} | "
      f"ABA {sum(m['route'] == 'ABA' for m in meta.values())} / "
      f"ABCA {sum(m['route'] == 'ABCA' for m in meta.values())} | "
      f"coverage scored {np.isfinite(cov_all).sum()}/{len(clips)} "
      f"(median {np.nanmedian(cov_all):.2f})")

# %%
# Trajectory-length bins: terciles of the step count, so the three bins are
# equal-sized by construction. Lengths reported in generation steps and in
# seconds of GT trajectory (steps x stride / fps). Bins can be empty while
# the battery is still collecting.
steps_all = np.array([meta[c]["n_steps"] for c in clips])
q1, q2 = np.percentile(steps_all, [100 / 3, 200 / 3])
BINS = {
    "short":  [c for c in clips if meta[c]["n_steps"] <= q1],
    "middle": [c for c in clips if q1 < meta[c]["n_steps"] <= q2],
    "long":   [c for c in clips if meta[c]["n_steps"] > q2],
}
for name, cs in BINS.items():
    if not cs:
        print(f"{name:6s}:   0 clips")
        continue
    s = np.array([meta[c]["n_steps"] for c in cs])
    print(f"{name:6s}: {len(cs):3d} clips | steps [{s.min()}..{s.max()}] "
          f"= [{s.min() * STRIDE / FPS:.1f}..{s.max() * STRIDE / FPS:.1f}] s")


In [ ]:
# %%
# (1)+(2) Per-bin comparison at NORMALIZED trajectory time: each clip's
# per-step metric is interpolated onto a shared unit-interval grid (using the
# stored `frac` covariate) and averaged across the bin's clips at each
# fractional timepoint; bands are 95% bootstrap CIs over clips (clips are the
# resampling unit — the paired design keeps arms comparable pointwise).
U = np.linspace(0.02, 1.0, 25)


def frac_curves(arm, cs, metric):
    return np.stack([np.interp(U, recs[arm][c]["frac"], recs[arm][c][metric]) for c in cs])


def boot_ci(mat, B=2000, seed=0):
    rng = np.random.default_rng(seed)
    bm = mat[rng.integers(0, mat.shape[0], (B, mat.shape[0]))].mean(axis=1)
    return np.percentile(bm, [2.5, 97.5], axis=0)


for bin_name, cs in BINS.items():
    if not cs:
        continue
    fig, axes = plt.subplots(3, 1, figsize=(9, 10), sharex=True)
    for ax, metric in zip(axes, METRICS):
        for arm in ARMS:
            mat = frac_curves(arm, cs, metric)
            mean = mat.mean(axis=0)
            lo, hi = boot_ci(mat)
            ax.plot(U, mean, color=COLORS[arm], lw=1.8,
                    label=f"{arm} (mean {np.mean([recs[arm][c][metric].mean() for c in cs]):.3f})")
            ax.fill_between(U, lo, hi, color=COLORS[arm], alpha=0.15, lw=0)
        ax.set_ylabel(metric)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
        ax.set_title(f"{metric} vs normalized trajectory time ({BETTER[metric]})", fontsize=10)
    axes[-1].set_xlabel("fraction of trajectory (0 = rollout start, 1 = loop closure)")
    plt.suptitle(f"{bin_name.upper()} trajectories ({len(cs)} clips)", fontsize=13)
    plt.tight_layout()
    plt.show()


In [ ]:
# %%
# (7) PAPER FIGURE — aggregated per-step metrics vs NORMALIZED trajectory time
# over ALL clips at once (views (1)/(2) split by length bin; this is the
# pooled curve). Mirrors the LoopNav paper figure so the two datasets'
# figures are visually identical: baselines gray/dashed, ours on Okabe-Ito
# hues with the flagship heaviest, legend in the PSNR panel only, band = 95%
# bootstrap CI of the MEAN with CLIPS as the resampling unit. Reuses this
# notebook's `clips` (so the ROUTE filter and the paired common-clip subset
# both apply) and the `U` / `frac_curves` / `boot_ci` helpers from cell (1).
#
# CAVEAT — the LEFT EDGE IS EXTRAPOLATION, not measurement. Each clip's curve
# is interpolated onto the shared grid from its `frac` covariate, and
# `np.interp` HOLDS the first value flat below the clip's own first step.
# SoundSpaces clips are short (a median of ~12 steps at stride 16), so most
# clips have no step at the first few grid points: that region is first-step
# values repeated. The printout reports the share per grid point; raise U_MIN
# to crop it. U_MIN = 0.02 is the loopnav-identical
# default, kept so both figures share an x-axis (loopnav clips are ~10x
# longer, where the same grid is fully populated).
import shutil

FIG_PATH = "results/figures/soundspaces_metrics_vs_progress.pdf"
U_MIN    = 0.02          # crop the flat-extrapolated left edge (0.02 = loopnav default)

# User-local TinyTeX for `text.usetex` (falls back to CM mathtext without it).
_tinytex = os.path.expanduser("~/.TinyTeX/bin/x86_64-linux")
if os.path.isdir(_tinytex) and _tinytex not in os.environ.get("PATH", ""):
    os.environ["PATH"] = _tinytex + os.pathsep + os.environ.get("PATH", "")

PRETTY = {"ss_temporal": "Temporal", "ss_worldmem": "WorldMem",
          "ss_pnp_meta": "FAR -- Meta", "ss_pnp_amz": "FAR -- Multi-Cue",
          "ssv2_pnp_meta": "FAR -- Meta", "ssv2_pnp_amz": "FAR -- Multi-Cue"}
STYLE = {  # (color, linestyle, linewidth); baselines FIRST -> ours overplot them
    "ss_temporal":   ("#9e9e9e", "--", 1.6),
    "ss_worldmem":   ("#4d4d4d", "-.", 1.6),
    "ss_pnp_meta":   ("#56B4E9", "-", 1.8),
    "ss_pnp_amz":    ("#D55E00", "-", 2.4),
    "ssv2_pnp_meta": ("#56B4E9", "-", 1.8),
    "ssv2_pnp_amz":  ("#D55E00", "-", 2.4),
}

_UF = U[U >= U_MIN]
_ns = np.array([meta[c]["n_steps"] for c in clips])
_first = np.array([recs[ARMS[0]][c]["frac"][0] for c in clips])
print(f"{len(clips)} clips | median {int(np.median(_ns))} steps "
      f"({np.median(_ns) * STRIDE / FPS:.1f} s of GT trajectory)")
print("flat-extrapolated share of clips at frac = "
      + ", ".join(f"{u:.2f}:{(_first > u).mean():.0%}" for u in _UF[:5]))

_ORDER = [a for a in STYLE if a in ARMS] + [a for a in ARMS if a not in STYLE]
_YLAB = {"psnr": r"PSNR $\uparrow$", "lpips": r"LPIPS $\downarrow$",
         "dreamsim": r"DreamSim $\downarrow$"}
rc = {"font.family": "serif", "mathtext.fontset": "cm", "font.size": 11,
      "axes.labelsize": 12, "legend.fontsize": 9}
if shutil.which("latex"):
    rc["text.usetex"] = True
with plt.rc_context(rc):
    fig, axes = plt.subplots(1, len(METRICS), figsize=(12, 3))
    for ax, metric in zip(axes, METRICS):
        for arm in _ORDER:
            col, ls, lw = STYLE.get(arm, (COLORS[arm], "-", 1.8))
            mat = frac_curves(arm, clips, metric)[:, U >= U_MIN]
            mean, (lo, hi) = mat.mean(axis=0), boot_ci(mat)
            ax.plot(_UF, mean, color=col, ls=ls, lw=lw,
                    label=PRETTY.get(arm, arm.replace("_", " ")))
            ax.fill_between(_UF, lo, hi, color=col, alpha=0.12, lw=0)
        ax.set_xlabel(r"Normalized Trajectory Time")
        ax.set_ylabel(_YLAB[metric])
        ax.set_xlim(0, 1)
        ax.grid(alpha=0.3, ls="--")
    axes[0].legend(loc="best", frameon=True, fancybox=False,
                   edgecolor="black", framealpha=1.0)
    fig.tight_layout()
    os.makedirs("results/figures", exist_ok=True)
    fig.savefig(FIG_PATH, bbox_inches="tight")
    plt.show()

print(f"\n{len(clips)} clips x {len(ARMS)} arms -> {FIG_PATH}")
for metric in METRICS:
    print(f"  {metric:9s} " + "  ".join(
        f"{PRETTY.get(a, a)}={frac_curves(a, clips, metric).mean():.3f}" for a in _ORDER))


In [ ]:
# %%
# Full-trajectory summary: mean of per-clip means (each clip weighted equally)
# with 95% bootstrap CIs, per bin and overall.
def summarize(cs, metric, arm):
    v = np.array([recs[arm][c][metric].mean() for c in cs])
    lo, hi = boot_ci(v[:, None])
    return v.mean(), float(lo[0]), float(hi[0])


for metric in METRICS:
    print(f"\n=== {metric} ({BETTER[metric]}) ===")
    header = f"{'arm':14s}" + "".join(f"{b:>22s}" for b in BINS) + f"{'ALL':>22s}"
    print(header)
    for arm in ARMS:
        row = f"{arm:14s}"
        for cs in list(BINS.values()) + [clips]:
            if not cs:
                row += f"{'-':>22s}"
                continue
            m, lo, hi = summarize(cs, metric, arm)
            row += f"{m:9.3f} [{lo:.3f},{hi:.3f}]"
        print(row)

# %%
# Box plots over ALL clips — the distribution behind the ALL column above.
# Row 1: per-clip full-trajectory means, one box per arm (box = IQR, line =
# median, x = mean; fliers hidden). Row 2: the SAME means with their 95%
# bootstrap CIs on their own axis — at large n the CI is ~1/sqrt(n) of the
# per-clip spread, so it is invisible at the box-plot scale and needs a
# zoomed panel to be compared across arms.
short = lambda a: a.replace("ss_", "")
fig, axes = plt.subplots(2, len(METRICS), figsize=(5.2 * len(METRICS), 8),
                         gridspec_kw=dict(height_ratios=[2, 1]), squeeze=False)
for j, metric in enumerate(METRICS):
    per_arm_v, stats = {}, {}
    for a in ARMS:
        v = np.array([recs[a][c][metric].mean() for c in clips])
        lo, hi = boot_ci(v[:, None])
        per_arm_v[a], stats[a] = v, (v.mean(), float(lo[0]), float(hi[0]))

    ax = axes[0, j]
    bp = ax.boxplot([per_arm_v[a] for a in ARMS], patch_artist=True, showfliers=False,
                    medianprops=dict(color="black"))
    for patch, a in zip(bp["boxes"], ARMS):
        patch.set_facecolor(COLORS[a]); patch.set_alpha(0.5)
    ax.plot(range(1, len(ARMS) + 1), [stats[a][0] for a in ARMS], "x", color="black",
            ms=7, mew=1.5, zorder=3, label="mean")
    ax.set_xticks(range(1, len(ARMS) + 1))
    ax.set_xticklabels([short(a) for a in ARMS], fontsize=8, rotation=20, ha="right")
    ax.set_title(f"{metric} per clip ({BETTER[metric]})", fontsize=10)
    ax.grid(alpha=0.3, axis="y")
    if j == 0:
        ax.legend(fontsize=8, loc="best")

    ax = axes[1, j]
    for k, a in enumerate(ARMS):
        m, lo, hi = stats[a]
        ax.errorbar([k], [m], yerr=[[m - lo], [hi - m]], fmt="o", ms=6, lw=2,
                    capsize=4, color=COLORS[a])
    ax.set_xticks(range(len(ARMS)))
    ax.set_xticklabels([short(a) for a in ARMS], fontsize=8, rotation=20, ha="right")
    ax.set_title(f"mean ± 95% bootstrap CI  (n={len(clips)} clips)", fontsize=9)
    ax.grid(alpha=0.3, axis="y")
plt.suptitle("Full-trajectory metrics aggregated over all clips", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# %%
# BETTER VIEW #1 — END-ALIGNED, REAL-TIME curves. Normalizing to the unit
# interval warps TIME: "progress 0.9" is ~1 s from closure on a short clip but
# ~10 s on a long one, yet drift accumulates in real seconds — so fractional
# averaging mixes different drift depths at every x. Aligning on steps BEFORE
# CLOSURE keeps the physics: every included clip contributes the same real-time
# window, and the loop-closure regime (where the retrieval mechanisms differ
# most) is measured composition-safely.
K = 25                                     # last K steps = K*stride/fps seconds
elig = [c for c in clips if meta[c]["n_steps"] >= K]
if not elig:
    print(f"no clips with >= {K} steps yet")
else:
    x = -np.arange(K)[::-1] * STRIDE / FPS     # seconds before closure (<= 0)
    fig, axes = plt.subplots(3, 1, figsize=(9, 10), sharex=True)
    for ax, metric in zip(axes, METRICS):
        for arm in ARMS:
            mat = np.stack([recs[arm][c][metric][-K:] for c in elig])
            mean, (lo, hi) = mat.mean(axis=0), boot_ci(mat)
            ax.plot(x, mean, color=COLORS[arm], lw=1.8, label=arm)
            ax.fill_between(x, lo, hi, color=COLORS[arm], alpha=0.15, lw=0)
        ax.set_ylabel(metric)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    axes[-1].set_xlabel(f"seconds before loop closure ({len(elig)} clips with >= {K} steps)")
    plt.suptitle("End-aligned (real-time) — the loop-closure approach", fontsize=13)
    plt.tight_layout()
    plt.show()


In [ ]:
# %%
# BETTER VIEW #2 — PAIRED per-clip differences vs the temporal baseline.
# Bin/arm means hide the pairing power of the design: every arm ran the SAME
# episodes with the SAME noise, so per-clip deltas cancel clip difficulty.
# Boxes show the per-clip distribution of (arm - temporal) full-trajectory
# means; annotations give the two-sided Wilcoxon signed-rank p-value.
BASE = next(a for a in ARMS if a.endswith("_temporal"))
others = [a for a in ARMS if a != BASE]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric in zip(axes, METRICS):
    data, labels, colors = [], [], []
    for bi, (bin_name, cs) in enumerate(BINS.items()):
        if not cs:
            continue
        base = np.array([recs[BASE][c][metric].mean() for c in cs])
        for arm in others:
            d = np.array([recs[arm][c][metric].mean() for c in cs]) - base
            data.append(d)
            labels.append(f"{bin_name}\n{arm.replace('ss_', '')}")
            colors.append(COLORS[arm])
    bp = ax.boxplot(data, patch_artist=True, showfliers=False)
    for patch, col in zip(bp["boxes"], colors):
        patch.set_facecolor(col)
        patch.set_alpha(0.5)
    for i, d in enumerate(data):
        p = wilcoxon(d).pvalue if len(d) > 1 and np.any(d != 0) else 1.0
        stars = "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else ""
        ax.text(i + 1, ax.get_ylim()[1], stars, ha="center", fontsize=11)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, fontsize=6, rotation=90)
    ax.set_title(f"Δ{metric} vs temporal ({BETTER[metric]})", fontsize=10)
    ax.grid(alpha=0.3, axis="y")
plt.suptitle("Paired per-clip differences vs temporal (Wilcoxon: * p<.05 ** p<.01 *** p<.001)")
plt.tight_layout()
plt.show()


In [ ]:
# %%
# Mechanism panel — retrieval diagnostics vs normalized progress: the mean
# metric distance of retrieved frames to the current pose, and the mean |Δt|
# of the picks. Divergent retrieval behaviour here EXPLAINS metric gaps above
# (e.g. geometry-anchored picks staying near the loop-closure region while
# audio picks cross occlusion boundaries).
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (key, name) in zip(axes, [("ret_dist", "mean pick distance to current pose (m)"),
                                  ("ret_dt", "mean pick |Δt| (s)")]):
    for arm in ARMS:
        mats = []
        for c in clips:
            r = recs[arm][c]
            if key not in r:
                continue
            v = np.abs(r[key]).mean(axis=1)
            mats.append(np.interp(U, r["frac"], v))
        if not mats:
            continue
        mat = np.stack(mats)
        mean, (lo, hi) = mat.mean(axis=0), boot_ci(mat)
        ax.plot(U, mean, color=COLORS[arm], lw=1.8, label=arm)
        ax.fill_between(U, lo, hi, color=COLORS[arm], alpha=0.15, lw=0)
    ax.set_xlabel("fraction of trajectory")
    ax.set_ylabel(name)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.suptitle("Retrieval diagnostics vs progress", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# %%
# SS-SPECIFIC — coverage dose-response of the audio channel. Per-clip PAIRED
# Δ(pnp_amz - pnp_meta) full-trajectory means (the same retriever with the
# audio cue on / off) against the clip's outbound->return view coverage.
# Prediction: the audio
# gain concentrates at LOW coverage (occluded pools, where geometric scoring
# cannot tell informative frames from occluded ones) and vanishes as
# coverage -> 1. Gray dots = clips; red points = coverage-bin means with 95%
# bootstrap CI; annotation = n and Wilcoxon p per bin.
_AMZ  = next(a for a in ARMS if a.endswith("_pnp_amz"))
_META = next(a for a in ARMS if a.endswith("_pnp_meta"))
cov = np.array([meta[c]["coverage"] for c in clips])
ok = np.isfinite(cov)
edges = [0.0, 0.3, 0.6, 1.01]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, metric in zip(axes, METRICS):
    d = np.array([recs[_AMZ][c][metric].mean() - recs[_META][c][metric].mean()
                  for c in clips])
    ax.scatter(cov[ok], d[ok], s=8, alpha=0.35, color="tab:gray")
    for a, b in zip(edges[:-1], edges[1:]):
        m = ok & (cov >= a) & (cov < b)
        if m.sum() == 0:
            continue
        v = d[m]
        lo, hi = boot_ci(v[:, None])
        p = wilcoxon(v).pvalue if len(v) > 4 and np.any(v != 0) else np.nan
        xc = (a + min(b, 1.0)) / 2
        ax.errorbar([xc], [v.mean()], yerr=[[v.mean() - lo[0]], [hi[0] - v.mean()]],
                    fmt="o", ms=7, capsize=4, color="tab:red", zorder=3)
        ax.annotate(f"n={m.sum()}\np={p:.3f}" if np.isfinite(p) else f"n={m.sum()}",
                    (xc, v.mean()), textcoords="offset points", xytext=(6, 8), fontsize=7)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xlabel("outbound->return view coverage")
    ax.set_ylabel(f"Δ{metric} (amz - meta)")
    sgn = "Δ<0 = audio helps" if metric != "psnr" else "Δ>0 = audio helps"
    ax.set_title(f"{metric} ({sgn})", fontsize=10)
    ax.grid(alpha=0.3)
plt.suptitle("Audio-channel gain vs pool coverage (paired Multi-Cue - Meta)", fontsize=12)
plt.tight_layout()
plt.show()

# Route split of the same paired delta (ABCA return legs traverse C->A
# stretches the outbound never saw — the extreme low-coverage regime).
print("\npaired amz - meta by route:")
for route in ("ABA", "ABCA"):
    cs = [c for c in clips if meta[c]["route"] == route]
    if not cs:
        continue
    row = f"{route:5s} n={len(cs):4d}  "
    for metric in METRICS:
        v = np.array([recs["ssv2_pnp_amz"][c][metric].mean()
                      - recs["ssv2_pnp_meta"][c][metric].mean() for c in cs])
        p = wilcoxon(v).pvalue if len(v) > 1 and np.any(v != 0) else np.nan
        row += f"Δ{metric} {v.mean():+.3f} (p={p:.3f})   "
    print(row)

# %% [markdown]
# Further views the records support without recollection: distance-to-closure
# alignment (`dist_to_closure` covariate — metrics vs METERS from the closure
# point); per-scene panels (Matterport scene id = first token of video_id);
# and support-STRATIFIED per-step curves once the per-frame observability npz
# is joined step-wise.
